# clip-grad-norm-pre-step — worked example 3: Clipping is a no-op when the norm is already small

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `clip-grad-norm-pre-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`clip_grad_norm_` only rescales when the global norm strictly exceeds `max_norm`. When the existing norm is at or below the threshold, the gradients are left untouched (bit-for-bit), though the function still returns the measured norm. This is why a generous `max_norm` rarely hurts well-behaved training.

## Worked solution

**Step 1 - set up small gradients.** We assign two parameters tiny grads whose combined global norm we can compute by hand: grads `[0.1, 0.1]` and `[0.1, 0.1]` give `sqrt(4 * 0.01) = 0.2`. We snapshot a clone of the grads so we can compare later.

**Step 2 - clip with a large max_norm.** Calling `clip_grad_norm_(params, max_norm=5.0)` measures the norm (0.2), sees that `0.2 <= 5.0`, and therefore does NOT rescale anything. The grads are identical to before.

**Step 3 - confirm no mutation.** We compare each grad against its saved clone with `torch.equal`. They match exactly, proving the no-op branch ran. The returned norm is still 0.2, useful for logging even when nothing was clipped.

**Why it works:** the threshold check `total > max_norm` gates the in-place multiply. Below threshold there is no multiply, so the optimizer sees the original gradient and clipping costs only a norm computation.

In [ ]:
import torch.nn.utils as nn_utils

t.manual_seed(0)

a = t.zeros(2, requires_grad=True)
b = t.zeros(2, requires_grad=True)
a.grad = t.full((2,), 0.1)
b.grad = t.full((2,), 0.1)

before = [a.grad.clone(), b.grad.clone()]
pre = nn_utils.clip_grad_norm_([a, b], max_norm=5.0)
unchanged = t.equal(a.grad, before[0]) and t.equal(b.grad, before[1])

print("pre-clip norm:", round(pre.item(), 4))
print("grads unchanged:", unchanged)